# Xenium cell-type annotation workflow: DCM and IHD

**Purpose.** Marker-guided cell-type annotation for the Xenium DCM and IHD datasets, from BANKSY-based spatial clustering to compartment-specific refinement.

Cells were grouped using BANKSY spatial clustering. Established marker genes were used to assign broad cardiomyocyte (CM), fibroblast (FB), immune and vascular classes. These assignments were refined by compartment-specific Scanpy/Leiden clustering and manual marker-based reannotation.

## Workflow and inputs

- **DCM:** initial BANKSY annotation and compartment-specific refinement; output: `DCM_final.h5ad` with `celltype_corrected`.
- **IHD:** initial BANKSY annotation and fibroblast-compartment refinement; output: `adata_celltype_final_corrected_v4.h5ad` with `celltype_final_corrected_v4`.
- The IHD fibroblast-refinement stage uses the annotated round-2 IHD object and the corresponding manual subcluster mappings.

Place `Clean_cells_DCM.csv`, `Clean_UMIs_DCM.csv`, `Clean_cells_IHD.csv`, `Clean_UMIs_IHD.csv`, and the annotated round-2 IHD object in the working directory. The spatial-analysis notebook also uses `Hs_Matrisome_Masterlist_Naba et al_2012.xlsx` for the broad ECM-marker screen.

**Dependencies:** Scanpy, AnnData, NumPy, pandas, BANKSY and its helper package, SciPy, statsmodels, Matplotlib, and openpyxl. Run code cells in order. Manual cluster mappings correspond to the Leiden settings specified in the notebook.


## Part A — DCM: build AnnData and perform initial BANKSY clustering

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData

from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition

RANDOM_SEED = 1234
DCM_CELLS = Path("Clean_cells_DCM.csv")
DCM_UMIS = Path("Clean_UMIs_DCM.csv")
DCM_FIRST_PASS = Path("adata_DCM_first_round.h5ad")

for file in (DCM_CELLS, DCM_UMIS):
    if not file.exists():
        raise FileNotFoundError(f"Required input is missing: {file.resolve()}")

sc.settings.verbosity = 2

### A1. Reconstruct cell-by-gene count matrix and retain spatial coordinates

In [ ]:
df_cells_dcm = pd.read_csv(DCM_CELLS)
df_umi_dcm = pd.read_csv(DCM_UMIS)

required_umi_cols = {"Cell_code", "Gene", "UMI"}
missing_umi_cols = required_umi_cols - set(df_umi_dcm.columns)
if missing_umi_cols:
    raise KeyError(f"Missing UMI columns: {sorted(missing_umi_cols)}")
if not {"x_centroid", "y_centroid"}.issubset(df_cells_dcm.columns):
    raise KeyError("Clean_cells_DCM.csv must contain x_centroid and y_centroid.")

expr_dcm = df_umi_dcm.pivot_table(
    index="Cell_code", columns="Gene", values="UMI", aggfunc="sum", fill_value=0
)
if "cell_id" in df_cells_dcm.columns:
    df_cells_dcm["cell_id"] = df_cells_dcm["cell_id"].astype(str)
    df_cells_dcm = df_cells_dcm.set_index("cell_id", drop=True)
else:
    df_cells_dcm.index = df_cells_dcm.index.astype(str)
    df_cells_dcm.index.name = "cell_id"
expr_dcm.index = expr_dcm.index.astype(str)

common_dcm = expr_dcm.index.intersection(df_cells_dcm.index)
if len(common_dcm) == 0:
    raise ValueError("No shared DCM cell IDs between count and coordinate tables.")
expr_dcm = expr_dcm.loc[common_dcm].copy()
df_cells_dcm = df_cells_dcm.loc[common_dcm].copy()
coords_dcm = df_cells_dcm[["x_centroid", "y_centroid"]].to_numpy(dtype=float)

adata = AnnData(
    X=expr_dcm.to_numpy(dtype=np.float32),
    obs=pd.DataFrame(index=expr_dcm.index.astype(str)),
    var=pd.DataFrame(index=expr_dcm.columns.astype(str)),
)
adata.layers["counts"] = adata.X.copy()
adata.obsm["coord_xy"] = coords_dcm
adata.obsm["spatial"] = coords_dcm
adata.obs["x_centroid"] = coords_dcm[:, 0]
adata.obs["y_centroid"] = coords_dcm[:, 1]
print("DCM AnnData before QC:", adata)

### A2. DCM quality-control thresholds used in the analysis workflow

In [ ]:
# DCM source workflow: minimum 10 counts and 10 detected genes;
# exclude the top 0.5% by counts or detected genes.
adata.obs["n_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs["n_genes"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()
count_low, gene_low = 10, 10
count_high = float(np.percentile(adata.obs["n_counts"], 99.5))
gene_high = float(np.percentile(adata.obs["n_genes"], 99.5))

adata.obs["keep_qc"] = (
    (adata.obs["n_counts"] >= count_low)
    & (adata.obs["n_counts"] <= count_high)
    & (adata.obs["n_genes"] >= gene_low)
    & (adata.obs["n_genes"] <= gene_high)
)
print({
    "n_before": int(adata.n_obs),
    "n_after": int(adata.obs["keep_qc"].sum()),
    "n_removed": int((~adata.obs["keep_qc"]).sum()),
    "n_counts_upper_99_5pct": count_high,
    "n_genes_upper_99_5pct": gene_high,
})
adata = adata[adata.obs["keep_qc"].to_numpy()].copy()
sc.pl.violin(adata, ["n_counts", "n_genes"], jitter=0.4, multi_panel=True)

### A3. BANKSY embedding and clustering (resolution 0.6)

In [ ]:
coord_keys = ("x_centroid", "y_centroid", "coord_xy")
banksy_dict = initialize_banksy(
    adata=adata,
    coord_keys=coord_keys,
    num_neighbours=15,
    nbr_weight_decay="scaled_gaussian",
    max_m=1,
    plt_edge_hist=False,
    plt_nbr_weights=False,
    plt_agf_angles=False,
    plt_theta=False,
)
banksy_dict, banksy_matrix = generate_banksy_matrix(
    adata, banksy_dict, lambda_list=[0.2], max_m=1
)
pca_umap(banksy_dict, pca_dims=[30], add_umap=True)

# Preserve the original resolution review (1.0, 1.2, 0.8, 0.6).
# Each resolution is evaluated on a copy so one clustering run does not
# overwrite the graph/labels used by another resolution.
import copy

banksy_resolution_results = {}
for resolution in [1.0, 1.2, 0.8, 0.6]:
    banksy_dict_resolution = copy.deepcopy(banksy_dict)
    result, _ = run_Leiden_partition(
        banksy_dict_resolution,
        resolutions=[resolution],
        num_nn=50,
        num_iterations=-1,
        partition_seed=RANDOM_SEED,
        match_labels=True,
        max_labels=50,
    )
    banksy_resolution_results[resolution] = result
    print(f"Resolution {resolution}: {result.index.tolist()}")

# The recorded manual cluster mapping is defined for resolution 0.6.
banksy_df_dcm = banksy_resolution_results[0.6]
adata_banksy = banksy_dict["scaled_gaussian"][0.2]["adata"]
adata.obsm["X_umap"] = adata_banksy.obsm["reduced_pc_30_umap"]
adata.obsm["spatial"] = np.asarray(adata.obsm["coord_xy"])
key_dcm = banksy_df_dcm.index[0]
labels_dcm = np.asarray(banksy_df_dcm.labels[key_dcm].dense).astype(int)
if len(labels_dcm) != adata.n_obs:
    raise ValueError("BANKSY cluster labels do not match DCM observations.")
adata.obs["banksy_cluster_0.6"] = pd.Categorical(labels_dcm)
print(adata.obs["banksy_cluster_0.6"].value_counts().sort_index())
sc.pl.embedding(adata, basis="X_umap", color="banksy_cluster_0.6", size=6,
                legend_loc="on data", title="DCM BANKSY clusters (resolution 0.6, UMAP)")
sc.pl.embedding(adata, basis="spatial", color="banksy_cluster_0.6", size=6,
                title="DCM BANKSY clusters (resolution 0.6, spatial)")

### A4. Review established markers and apply the recorded broad cluster mapping

In [ ]:
marker_genes = {
    "CM": ["TNNT2", "MYH6", "MYH7"],
    "FB": ["COL3A1", "LUM", "PDGFRA", "DDR2", "HAPLN1", "VCAN"],
    "Endo": ["PECAM1", "VWF", "ACKR1"],
    "Myeloid": ["CD68", "ADGRE1"],
    "Tcell": ["CD3D", "CD3E", "TRAC"],
    "Bcell": ["MS4A1", "CD79A"],
    "Adipocyte": ["ADIPOQ", "PLIN1", "FABP4", "LIPE"],
    "Vascular_SMC": ["ACTA2", "TAGLN", "MYH11"],
}
marker_list = sorted({g for genes in marker_genes.values() for g in genes if g in adata.var_names})
adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)
if marker_list:
    sc.pl.dotplot(adata_norm, marker_list, groupby="banksy_cluster_0.6",
                  standard_scale="var", cmap="Reds")

cluster_to_celltype_dcm = {
    0: "vascular", 1: "FB", 2: "CM", 3: "FB", 4: "CM",
    5: "vascular", 6: "CM", 7: "vascular", 8: "CM", 9: "CM",
    10: "immune", 11: "vascular", 12: "vascular", 13: "immune",
    14: "vascular", 15: "immune", 16: "FB", 17: "vascular", 18: "immune",
}
clusters_dcm = adata.obs["banksy_cluster_0.6"].astype(int)
adata.obs["celltype"] = clusters_dcm.map(cluster_to_celltype_dcm)
if adata.obs["celltype"].isna().any():
    raise ValueError("At least one DCM BANKSY cluster is missing a manual mapping.")
adata.obs["celltype"] = adata.obs["celltype"].astype("category")
print(adata.obs["celltype"].value_counts())
sc.pl.umap(adata, color="celltype")
sc.pl.embedding(adata, basis="spatial", color="celltype", size=6)
adata.write_h5ad(DCM_FIRST_PASS, compression="gzip")
print(f"Saved broad DCM annotation: {DCM_FIRST_PASS.resolve()}")

## Part B — DCM: iterative compartment-specific Leiden refinement

The next section contains the DCM refinement code, including the marker panels, compartment-specific clustering and the manual subcluster-to-cell-type mappings. The notebook retains the marker panels, compartment-specific clustering and manual subcluster-to-cell-type mappings used for the final annotation. Run the section in order after the DCM first-pass cells above. The final object written by the source workflow is `DCM_final.h5ad`.

In [ ]:
import scanpy as sc

adata = sc.read("adata_DCM_first_round.h5ad")

In [ ]:
print(adata)

In [ ]:
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
fb_sub = adata[adata.obs["celltype"] == "FB"].copy()
fb_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(fb_sub, target_sum=1e4)
sc.pp.log1p(fb_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(fb_sub, n_comps=30)
sc.pp.neighbors(fb_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(fb_sub, resolution=2.0, random_state=0, flavor="leidenalg")
sc.tl.umap(fb_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], fb_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], fb_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], fb_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], fb_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring ---
fb_sub.obs["CM_mean"] = np.array(fb_sub[:, cm_markers].X.mean(axis=1)).flatten()
fb_sub.obs["FB_mean"] = np.array(fb_sub[:, fb_markers].X.mean(axis=1)).flatten()
fb_sub.obs["Vasc_mean"] = np.array(fb_sub[:, vasc_markers].X.mean(axis=1)).flatten()
fb_sub.obs["Immune_mean"] = np.array(fb_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores ---
sc.tl.score_genes(fb_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(fb_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(fb_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(fb_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (FB-focused) ---
fb_sub.obs["FB_minus_CM"] = fb_sub.obs["FB_mean"] - fb_sub.obs["CM_mean"]
fb_sub.obs["FB_minus_Vasc"] = fb_sub.obs["FB_mean"] - fb_sub.obs["Vasc_mean"]
fb_sub.obs["FB_minus_Immune"] = fb_sub.obs["FB_mean"] - fb_sub.obs["Immune_mean"]

fb_sub.obs["CM_minus_FB"] = fb_sub.obs["CM_mean"] - fb_sub.obs["FB_mean"]
fb_sub.obs["Vasc_minus_FB"] = fb_sub.obs["Vasc_mean"] - fb_sub.obs["FB_mean"]
fb_sub.obs["Immune_minus_FB"] = fb_sub.obs["Immune_mean"] - fb_sub.obs["FB_mean"]

# --- cluster summary ---
fb_scores = (
    fb_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "FB_minus_CM", "FB_minus_Vasc", "FB_minus_Immune"
    ]]
    .mean()
    .sort_values("FB_minus_CM", ascending=False)
)

print(fb_scores.round(3))

# --- visualization ---
sc.pl.umap(
    fb_sub,
    color=[
        "leiden",
        "FB_mean",
        "CM_mean",
        "Vasc_mean",
        "Immune_mean",
        "FB_minus_CM",
        "FB_minus_Vasc"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================
# Marker sets (FB context)
# =========================
markers = {
    "FB_core": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "ECM_remodeling": ["MMP2", "TIMP1", "TIMP2", "VCAN"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

marker_list = []
for genes in markers.values():
    marker_list += present(genes, fb_sub)

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    fb_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="FB subset: marker genes (FB / SMC / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    fb_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    fb_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="FB subset: top ranked genes per cluster",
    dendrogram=False
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = fb_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    fb_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="FB subset: markers + ranked genes",
    dendrogram=False
)

In [ ]:
sc.tl.rank_genes_groups(
    fb_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    fb_sub,
    sharey=False,
    title="Ranked genes"
)

In [ ]:
import scanpy as sc
import pandas as pd

# =========================================================
# Marker sets for fibroblast subclustering
# =========================================================
markers = {
    "FB_core": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "ECM_remodeling": ["MMP2", "TIMP1", "TIMP2", "VCAN"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=20):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, random_state=0, flavor="leidenalg")
    sc.tl.umap(sub)

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    print("\nSubcluster sizes:")
    print(sub.obs["leiden"].value_counts())

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: FB vs SMC vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes nur wenn >1 Subcluster
    # -------------------------
    n_subclusters = sub.obs["leiden"].nunique()

    if n_subclusters >= 2:
        sc.tl.rank_genes_groups(
            sub,
            groupby="leiden",
            method="wilcoxon",
            n_genes=n_ranked
        )

        sc.pl.rank_genes_groups_dotplot(
            sub,
            n_genes=5,
            standard_scale="var",
            cmap="Reds",
            title=f"Cluster {cluster_id}: top genes",
            dendrogram=False
        )

        ranked = sc.get.rank_genes_groups_df(sub, group=None)

        print("\n=== Top ranked genes per subcluster ===")
        for scl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
            top_cl = ranked[ranked["group"] == scl].head(n_ranked)
            print(f"\nSubcluster {scl}")
            print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    else:
        ranked = None
        print("\nNur ein Subcluster gefunden -> keine ranked genes zwischen Subclustern moeglich.")

    return sub, ranked

In [ ]:
# =========================================================
# Clusters selected for review within the fibroblast compartment
# Cluster 14 is directly reannotated as CM and is therefore not subclustered here
# Clusters 16/17 are directly reannotated as vascular and are therefore not subclustered here
# =========================================================
clusters_to_check = ["1", "2", "3", "6", "7", "9", "11", "15"]

cluster_resolutions = {
    "1": 0.8,
    "2": 0.7,
    "3": 0.8,
    "6": 0.6,
    "7": 0.7,
    "9": 0.7,
    "11": 0.8,
    "15": 0.6,
}

subclusters = {}
ranked_tables = {}

fb_sub.obs["leiden"] = fb_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(fb_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in fb_sub.obs.columns:
        fb_sub.obs[subcol] = pd.Series(index=fb_sub.obs_names, dtype="object")

    fb_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
print(adata)

In [ ]:
adata.write("DCM_fb_clean.h5ad")

In [ ]:
adata = sc.read("DCM_fb_clean.h5ad")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
vasc_sub = adata[adata.obs["celltype_corrected"] == "vascular"].copy()
vasc_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(vasc_sub, target_sum=1e4)
sc.pp.log1p(vasc_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(vasc_sub, n_comps=30)
sc.pp.neighbors(vasc_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(vasc_sub, resolution=2.0, random_state=0, flavor="leidenalg")
sc.tl.umap(vasc_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], vasc_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], vasc_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], vasc_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], vasc_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring (recommended for Xenium) ---
vasc_sub.obs["CM_mean"] = np.array(vasc_sub[:, cm_markers].X.mean(axis=1)).flatten()
vasc_sub.obs["FB_mean"] = np.array(vasc_sub[:, fb_markers].X.mean(axis=1)).flatten()
vasc_sub.obs["Vasc_mean"] = np.array(vasc_sub[:, vasc_markers].X.mean(axis=1)).flatten()
vasc_sub.obs["Immune_mean"] = np.array(vasc_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores (secondary, not primary) ---
sc.tl.score_genes(vasc_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(vasc_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(vasc_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(vasc_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (vascular-focused) ---
vasc_sub.obs["Vasc_minus_CM"] = vasc_sub.obs["Vasc_mean"] - vasc_sub.obs["CM_mean"]
vasc_sub.obs["Vasc_minus_FB"] = vasc_sub.obs["Vasc_mean"] - vasc_sub.obs["FB_mean"]
vasc_sub.obs["Vasc_minus_Immune"] = vasc_sub.obs["Vasc_mean"] - vasc_sub.obs["Immune_mean"]
vasc_sub.obs["FB_minus_Vasc"] = vasc_sub.obs["FB_mean"] - vasc_sub.obs["Vasc_mean"]
vasc_sub.obs["CM_minus_Vasc"] = vasc_sub.obs["CM_mean"] - vasc_sub.obs["Vasc_mean"]
vasc_sub.obs["Immune_minus_Vasc"] = vasc_sub.obs["Immune_mean"] - vasc_sub.obs["Vasc_mean"]

# --- cluster summary ---
vasc_scores = (
    vasc_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "Vasc_minus_CM", "Vasc_minus_FB", "Vasc_minus_Immune"
    ]]
    .mean()
    .sort_values("Vasc_minus_CM", ascending=False)
)

print(vasc_scores.round(3))

# --- visualization ---
sc.pl.umap(
    vasc_sub,
    color=[
        "leiden",
        "Vasc_mean",
        "CM_mean",
        "FB_mean",
        "Immune_mean",
        "Vasc_minus_CM",
        "Vasc_minus_FB"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================
# Marker sets (vascular context)
# =========================
markers = {
    "Vascular_core": ["PECAM1", "VWF", "CD93"],
    "Endothelial_activation": ["APOLD1", "RAMP2", "SOX18", "MMRN2"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"]
}

marker_list = sum(markers.values(), [])

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    vasc_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Vascular subset: marker genes (endo / perivascular / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    vasc_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    vasc_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="Vascular subset: top ranked genes per cluster"
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = vasc_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    vasc_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Vascular subset: markers + ranked genes"
)

In [ ]:
sc.tl.rank_genes_groups(
    vasc_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    vasc_sub,
    sharey=False,
    title="Ranked genes"
)

In [ ]:
import scanpy as sc
import pandas as pd

markers = {
    "Vascular_core": ["PECAM1", "VWF", "CD93"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=15):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, random_state=0, flavor="leidenalg")
    sc.tl.umap(sub)

    # Recompute the dendrogram for this subset
    sc.tl.dendrogram(sub, groupby="leiden")

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: vascular vs SMC vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes
    # -------------------------
    sc.tl.rank_genes_groups(
        sub,
        groupby="leiden",
        method="wilcoxon",
        n_genes=n_ranked
    )

    # Plot ranked genes
    sc.pl.rank_genes_groups_dotplot(
        sub,
        n_genes=5,
        standard_scale="var",
        cmap="Reds",
        title=f"Cluster {cluster_id}: top genes",
        dendrogram=False
    )

    # -------------------------
    # Tabelle mit ranked genes pro Subcluster
    # -------------------------
    ranked = sc.get.rank_genes_groups_df(sub, group=None)

    print("\n=== Top ranked genes per subcluster ===")
    for cl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
        top_cl = ranked[ranked["group"] == cl].head(n_ranked)
        print(f"\nSubcluster {cl}")
        print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    return sub, ranked

In [ ]:
clusters_to_check = [
    "2", "3", "4", "6", "9", "10", "12", "13",
    "14", "15", "16", "17", "18", "19", "20", "21", "22"
]

cluster_resolutions = {
    "2": 0.8,
    "3": 0.8,
    "4": 0.7,
    "6": 0.8,
    "9": 0.7,
    "10": 0.8,
    "12": 0.7,
}
subclusters = {}
ranked_tables = {}

vasc_sub.obs["leiden"] = vasc_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    # analyze_cluster gibt vermutlich (sub, ranked) zurück
    sub, ranked = analyze_cluster(vasc_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    # Spalte nur einmal anlegen, sonst überschreibst du sie in jeder Runde neu
    if subcol not in vasc_sub.obs.columns:
        vasc_sub.obs[subcol] = pd.Series(index=vasc_sub.obs_names, dtype="object")

    # Subcluster zurückschreiben
    vasc_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
clusters_to_check = ["13", "14", "15", "16", "17", "18", "19"
]

cluster_resolutions = {
    "13": 0.6,
    "14": 0.7,
    "15": 0.5,
    "16": 0.7,
    "17": 0.7,
    "18": 0.6,
}
subclusters = {}
ranked_tables = {}

vasc_sub.obs["leiden"] = vasc_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    # analyze_cluster gibt vermutlich (sub, ranked) zurück
    sub, ranked = analyze_cluster(vasc_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    # Spalte nur einmal anlegen, sonst überschreibst du sie in jeder Runde neu
    if subcol not in vasc_sub.obs.columns:
        vasc_sub.obs[subcol] = pd.Series(index=vasc_sub.obs_names, dtype="object")

    # Subcluster zurückschreiben
    vasc_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
clusters_to_check = ["19", "20", "21", "22"
]

cluster_resolutions = {
    "19": 0.5,
    "20": 0.5,
    "21": 0.6,
    "22": 0.5,
}
subclusters = {}
ranked_tables = {}

vasc_sub.obs["leiden"] = vasc_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    # analyze_cluster gibt vermutlich (sub, ranked) zurück
    sub, ranked = analyze_cluster(vasc_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    # Spalte nur einmal anlegen, sonst überschreibst du sie in jeder Runde neu
    if subcol not in vasc_sub.obs.columns:
        vasc_sub.obs[subcol] = pd.Series(index=vasc_sub.obs_names, dtype="object")

    # Subcluster zurückschreiben
    vasc_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
# =========================================================
# 0) Preparation
# =========================================================
vasc_sub.obs["leiden"] = vasc_sub.obs["leiden"].astype(str)

# Initialize all cells as vascular before applying the manual mapping
vasc_sub.obs["celltype_vascular_refined"] = "vascular"

# =========================================================
# 1) Subcluster-Mappings
# =========================================================
vascular_maps = {
    "2": {
        "0": "CM",
        "1": "contaminated",
        "2": "vascular",
        "3": "vascular",
        "4": "vascular",
    },
    "3": {
        "0": "vascular",
        "1": "contaminated",
        "2": "vascular",
        "3": "CM",
        "4": "FB",
        "5": "immune",
    },
    "4": {
        "0": "FB",
        "1": "vascular",
        "2": "vascular",
        "3": "FB",
        "4": "contaminated",
        "5": "immune",
    },
    "6": {
        "0": "vascular",
        "1": "contaminated",
        "2": "CM",
        "3": "CM",
        "4": "contaminated",
        "5": "contaminated",
    },
    "9": {
        "0": "vascular",
        "1": "contaminated",
        "2": "FB",
        "3": "vascular",
        "4": "vascular",
    },
    "10": {
        "0": "contaminated",
        "1": "vascular",
        "2": "vascular",
        "3": "vascular",
        "4": "CM",
    },
    "12": {
        "0": "vascular",
        "1": "vascular",
        "2": "CM",
        "3": "FB",
        "4": "vascular",
    },
    "13": {
        "0": "contaminated",
        "1": "vascular",
        "2": "CM",
        "3": "vascular",
    },
    "14": {
        "0": "contaminated",
        "1": "CM",
        "2": "vascular",
        "3": "contaminated",
        "4": "vascular",
    },
    "15": {
        "0": "contaminated",
        "1": "FB",
        "2": "CM",
        "3": "vascular",
        "4": "immune",
    },
    "16": {
        "0": "vascular",
        "1": "FB",
        "2": "CM",
    },
    "17": {
        "0": "vascular",
        "1": "contaminated",
        "2": "contaminated",
        "3": "CM",
    },
    "18": {
        "0": "CM",
        "1": "FB",
        "2": "contaminated",
        "3": "CM",
        "4": "contaminated",
    },
    "19": {
        "0": "FB",
        "1": "vascular",
        "2": "immune",
        "3": "contaminated",
    },
    "20": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "vascular",
        "3": "vascular",
    },
    "21": {
        "0": "contaminated",
        "1": "FB",
        "2": "contaminated",
        "3": "vascular",
    },
    "22": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "contaminated",
        "3": "contaminated",
    },
}

# =========================================================
# 2) Subcluster-Reannotation anwenden
# =========================================================
for main_clust, mapping in vascular_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = vasc_sub.obs["leiden"] == main_clust

    if subcol not in vasc_sub.obs.columns:
        print(f"Warning: {subcol} not found. Skipping cluster {main_clust}.")
        continue

    vasc_sub.obs.loc[mask, "celltype_vascular_refined"] = (
        vasc_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# =========================================================
# 3) Kontrolle in vasc_sub
# =========================================================
print("=== Hauptcluster vs. vascular refined annotation ===")
print(pd.crosstab(vasc_sub.obs["leiden"], vasc_sub.obs["celltype_vascular_refined"]))

print("\n=== Finale Annotationen in vasc_sub ===")
print(vasc_sub.obs["celltype_vascular_refined"].value_counts(dropna=False))

# =========================================================
# 4) Auf adata übertragen
# (WICHTIG: bestehende celltype_corrected NICHT überschreiben!)
# =========================================================
common_cells = adata.obs_names.intersection(vasc_sub.obs_names)

adata.obs.loc[common_cells, "celltype_corrected"] = (
    vasc_sub.obs.loc[common_cells, "celltype_vascular_refined"].astype(str)
)

# =========================================================
# 5) Kategorien setzen
# =========================================================
adata.obs["celltype_corrected"] = pd.Categorical(
    adata.obs["celltype_corrected"].astype(str),
    categories=[
        "CM",
        "FB",
        "vascular",
        "immune",
        "contaminated",
    ],
)

# =========================================================
# 6) Kontrolle in adata
# =========================================================
print("\n=== Verteilung in adata nach vascular-Korrektur ===")
print(adata.obs["celltype_corrected"].value_counts(dropna=False))

print("\n=== Vergleich vorher vs. nach vascular-Korrektur ===")
print(pd.crosstab(
    adata.obs.loc[common_cells, "celltype"],
    adata.obs.loc[common_cells, "celltype_corrected"]
))

In [ ]:
adata.write("DCM_fb_vasc_clean.h5ad")

In [ ]:
adata = sc.read("DCM_fb_vasc_clean.h5ad")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# =========================================================
# --- subset ---
# =========================================================
cm_sub = adata[adata.obs["celltype_corrected"] == "CM"].copy()
cm_sub

# =========================================================
# --- preprocessing (Xenium-friendly) ---
# =========================================================
sc.pp.normalize_total(cm_sub, target_sum=1e4)
sc.pp.log1p(cm_sub)

# =========================================================
# --- clustering (NO HVG, NO scale) ---
# =========================================================
sc.tl.pca(cm_sub, n_comps=30)
sc.pp.neighbors(cm_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(cm_sub, resolution=2.0, random_state=0, flavor="leidenalg")
sc.tl.umap(cm_sub)

# =========================================================
# --- marker sets ---
# =========================================================
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], cm_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], cm_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], cm_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], cm_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# =========================================================
# --- expression-based scoring ---
# =========================================================
cm_sub.obs["CM_mean"] = np.array(cm_sub[:, cm_markers].X.mean(axis=1)).flatten()
cm_sub.obs["FB_mean"] = np.array(cm_sub[:, fb_markers].X.mean(axis=1)).flatten()
cm_sub.obs["Vasc_mean"] = np.array(cm_sub[:, vasc_markers].X.mean(axis=1)).flatten()
cm_sub.obs["Immune_mean"] = np.array(cm_sub[:, imm_markers].X.mean(axis=1)).flatten()

# =========================================================
# --- optional scanpy scores ---
# =========================================================
sc.tl.score_genes(cm_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(cm_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(cm_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(cm_sub, imm_markers, score_name="Immune_score")

# =========================================================
# --- derived metrics (CM-focused) ---
# =========================================================
cm_sub.obs["CM_minus_FB"] = cm_sub.obs["CM_mean"] - cm_sub.obs["FB_mean"]
cm_sub.obs["CM_minus_Vasc"] = cm_sub.obs["CM_mean"] - cm_sub.obs["Vasc_mean"]
cm_sub.obs["CM_minus_Immune"] = cm_sub.obs["CM_mean"] - cm_sub.obs["Immune_mean"]

cm_sub.obs["FB_minus_CM"] = cm_sub.obs["FB_mean"] - cm_sub.obs["CM_mean"]
cm_sub.obs["Vasc_minus_CM"] = cm_sub.obs["Vasc_mean"] - cm_sub.obs["CM_mean"]
cm_sub.obs["Immune_minus_CM"] = cm_sub.obs["Immune_mean"] - cm_sub.obs["CM_mean"]

# =========================================================
# --- cluster summary ---
# =========================================================
cm_scores = (
    cm_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "CM_minus_FB", "CM_minus_Vasc", "CM_minus_Immune"
    ]]
    .mean()
    .sort_values("CM_minus_FB", ascending=False)
)

print(cm_scores.round(3))

# =========================================================
# --- visualization ---
# =========================================================
sc.pl.umap(
    cm_sub,
    color=[
        "leiden",
        "CM_mean",
        "FB_mean",
        "Vasc_mean",
        "Immune_mean",
        "CM_minus_FB",
        "CM_minus_Vasc"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================
# Marker sets (CM context)
# =========================
markers = {
    "CM_core": ["TNNT2", "MYH6", "MYH7", "TTN", "DES"],
    "CM_stress_DCM": ["NPPA", "NPPB", "S100A1"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA", "COL3A1"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93", "CD34", "CLEC14A"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"],
}

marker_list = sum(markers.values(), [])

# Keep only genes present in the panel
marker_list = [g for g in marker_list if g in cm_sub.var_names]

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    cm_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="CM subset: marker genes (CM / stress / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    cm_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    cm_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="CM subset: top ranked genes per cluster"
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = cm_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(dict.fromkeys(ranked.flatten()))[:30]

combined_genes = list(dict.fromkeys(marker_list + top_ranked))
combined_genes = [g for g in combined_genes if g in cm_sub.var_names]

sc.pl.dotplot(
    cm_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="CM subset: markers + ranked genes"
)

In [ ]:
sc.tl.rank_genes_groups(
    cm_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    cm_sub,
    sharey=False,
    title="Ranked genes"
)

In [ ]:
import scanpy as sc
import pandas as pd

# =========================
# Marker sets (CM context)
# =========================
markers = {
    "CM_core": ["TNNT2", "MYH6", "MYH7", "TTN", "DES"],
    "CM_stress_DCM": ["NPPA", "NPPB", "S100A1", "FHL2", "GJA1"],
    "CM_metabolic_state": ["ATP1A3", "PLIN4", "LPL", "PTGDS", "SMYD2"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA", "COL3A1"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93", "CD34", "CLEC14A"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"],
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=15):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, random_state=0, flavor="leidenalg")
    sc.tl.umap(sub)

    # Recompute the dendrogram for this subset
    sc.tl.dendrogram(sub, groupby="leiden")

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    # Duplikate entfernen, Reihenfolge behalten
    marker_list = list(dict.fromkeys(marker_list))

    print("Markers used:", marker_list)

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: CM vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes
    # -------------------------
    sc.tl.rank_genes_groups(
        sub,
        groupby="leiden",
        method="wilcoxon",
        n_genes=n_ranked
    )

    sc.pl.rank_genes_groups_dotplot(
        sub,
        n_genes=5,
        standard_scale="var",
        cmap="Reds",
        title=f"Cluster {cluster_id}: top genes",
        dendrogram=False
    )

    # -------------------------
    # Tabelle mit ranked genes pro Subcluster
    # -------------------------
    ranked = sc.get.rank_genes_groups_df(sub, group=None)

    print("\n=== Top ranked genes per subcluster ===")
    for cl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
        top_cl = ranked[ranked["group"] == cl].head(n_ranked)
        print(f"\nSubcluster {cl}")
        print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    return sub, ranked

In [ ]:
clusters_to_check = [
    "6", "12", "14", "15"
]

cluster_resolutions = {
    "6": 0.7,
    "12": 0.7,
    "14": 0.5,
    "15": 0.6,
}

subclusters = {}
ranked_tables = {}

cm_sub.obs["leiden"] = cm_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(cm_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in cm_sub.obs.columns:
        cm_sub.obs[subcol] = pd.Series(index=cm_sub.obs_names, dtype="object")

    cm_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
clusters_to_check = [
 "16", "18", "19"
]

cluster_resolutions = {
    "16": 0.7,
    "18": 0.6,
    "19": 0.5,
}

subclusters = {}
ranked_tables = {}

cm_sub.obs["leiden"] = cm_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(cm_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in cm_sub.obs.columns:
        cm_sub.obs[subcol] = pd.Series(index=cm_sub.obs_names, dtype="object")

    cm_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
# =========================================================
# 0) Preparation
# =========================================================
cm_sub.obs["leiden"] = cm_sub.obs["leiden"].astype(str)

# Initialize all cells as CM before applying the manual mapping
cm_sub.obs["celltype_CM_refined"] = "CM"

# =========================================================
# 1) Subcluster-Mappings
# =========================================================
cm_maps = {
    "6": {
        "0": "vascular",
        "1": "CM",
        "2": "FB",
        "3": "CM",
    },
    "12": {
        "0": "CM",
        "1": "contaminated",
        "2": "contaminated",
        "3": "vascular",
    },
    "14": {
        "0": "CM",
        "1": "contaminated",
        "2": "contaminated",
        "3": "contaminated",
    },
    "15": {
        "0": "CM",
        "1": "contaminated",
        "2": "contaminated",
        "3": "immune",
    },
    "16": {
        "0": "CM",
        "1": "CM",
        "2": "vascular",
        "3": "contaminated",
    },
    "18": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "contaminated",
        "3": "vascular",
    },
    "19": {
        "0": "contaminated",
        "1": "CM",
        "2": "contaminated",
    },
}

# =========================================================
# 2) Subcluster-Reannotation anwenden
# =========================================================
for main_clust, mapping in cm_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = cm_sub.obs["leiden"] == main_clust

    if subcol not in cm_sub.obs.columns:
        print(f"Warning: {subcol} not found. Skipping cluster {main_clust}.")
        continue

    cm_sub.obs.loc[mask, "celltype_CM_refined"] = (
        cm_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# =========================================================
# 3) Direkte Hauptcluster-Reannotation
# =========================================================
cm_sub.obs.loc[cm_sub.obs["leiden"] == "9", "celltype_CM_refined"] = "FB"
cm_sub.obs.loc[cm_sub.obs["leiden"] == "10", "celltype_CM_refined"] = "vascular"
cm_sub.obs.loc[cm_sub.obs["leiden"] == "17", "celltype_CM_refined"] = "vascular"

# =========================================================
# 4) Kontrolle in cm_sub
# =========================================================
print("=== Hauptcluster vs. CM refined annotation ===")
print(pd.crosstab(cm_sub.obs["leiden"], cm_sub.obs["celltype_CM_refined"]))

print("\n=== Finale Annotationen in cm_sub ===")
print(cm_sub.obs["celltype_CM_refined"].value_counts(dropna=False))

# =========================================================
# 5) Auf adata übertragen
# WICHTIG: bestehende celltype_corrected NICHT neu initialisieren
# =========================================================
common_cells = adata.obs_names.intersection(cm_sub.obs_names)

adata.obs.loc[common_cells, "celltype_corrected"] = (
    cm_sub.obs.loc[common_cells, "celltype_CM_refined"].astype(str)
)

# =========================================================
# 6) Kategorien setzen
# =========================================================
adata.obs["celltype_corrected"] = pd.Categorical(
    adata.obs["celltype_corrected"].astype(str),
    categories=[
        "CM",
        "FB",
        "vascular",
        "immune",
        "contaminated",
    ],
)

# =========================================================
# 7) Kontrolle in adata
# =========================================================
print("\n=== Verteilung in adata nach CM-Korrektur ===")
print(adata.obs["celltype_corrected"].value_counts(dropna=False))

print("\n=== Vergleich vorher vs. nach CM-Korrektur ===")
print(pd.crosstab(
    adata.obs.loc[common_cells, "celltype"],
    adata.obs.loc[common_cells, "celltype_corrected"]
))

In [ ]:
adata.write("adata_DCM_fb_vasc_cm_clean.h5ad")

In [ ]:
adata = sc.read("adata_DCM_fb_vasc_cm_clean.h5ad")

In [ ]:
print(adata)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
immune_sub = adata[adata.obs["celltype_corrected"] == "immune"].copy()
immune_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(immune_sub, target_sum=1e4)
sc.pp.log1p(immune_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(immune_sub, n_comps=30)
sc.pp.neighbors(immune_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(immune_sub, resolution=1.4, random_state=0, flavor="leidenalg")
sc.tl.umap(immune_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], immune_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], immune_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], immune_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], immune_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring ---
immune_sub.obs["CM_mean"] = np.array(immune_sub[:, cm_markers].X.mean(axis=1)).flatten()
immune_sub.obs["FB_mean"] = np.array(immune_sub[:, fb_markers].X.mean(axis=1)).flatten()
immune_sub.obs["Vasc_mean"] = np.array(immune_sub[:, vasc_markers].X.mean(axis=1)).flatten()
immune_sub.obs["Immune_mean"] = np.array(immune_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores ---
sc.tl.score_genes(immune_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(immune_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(immune_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(immune_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (immune-focused) ---
immune_sub.obs["Immune_minus_CM"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["CM_mean"]
immune_sub.obs["Immune_minus_FB"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["FB_mean"]
immune_sub.obs["Immune_minus_Vasc"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["Vasc_mean"]

immune_sub.obs["CM_minus_Immune"] = immune_sub.obs["CM_mean"] - immune_sub.obs["Immune_mean"]
immune_sub.obs["FB_minus_Immune"] = immune_sub.obs["FB_mean"] - immune_sub.obs["Immune_mean"]
immune_sub.obs["Vasc_minus_Immune"] = immune_sub.obs["Vasc_mean"] - immune_sub.obs["Immune_mean"]

# --- cluster summary ---
immune_scores = (
    immune_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "Immune_minus_CM", "Immune_minus_FB", "Immune_minus_Vasc"
    ]]
    .mean()
    .sort_values("Immune_minus_CM", ascending=False)
)

print(immune_scores.round(3))

# --- visualization ---
sc.pl.umap(
    immune_sub,
    color=[
        "leiden",
        "Immune_mean",
        "CM_mean",
        "FB_mean",
        "Vasc_mean",
        "Immune_minus_CM",
        "Immune_minus_Vasc"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================
# Marker sets (immune context)
# =========================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "NKG7", "CD68", "FCGR3A", "MPEG1"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

marker_list = []
for genes in markers.values():
    marker_list += present(genes, immune_sub)

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    immune_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Immune subset: marker genes (immune / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    immune_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    immune_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="Immune subset: top ranked genes per cluster",
    dendrogram=False
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = immune_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    immune_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Immune subset: markers + ranked genes",
    dendrogram=False
)

In [ ]:
sc.tl.rank_genes_groups(
    immune_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    immune_sub,
    sharey=False,
    title="Ranked genes"
)

In [ ]:
import scanpy as sc
import pandas as pd

# =========================================================
# Marker sets for immune-cell subclustering
# =========================================================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "NKG7", "CD68", "FCGR3A", "MPEG1"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=15):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, random_state=0, flavor="leidenalg")
    sc.tl.umap(sub)

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    print("\nSubcluster sizes:")
    print(sub.obs["leiden"].value_counts())

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: immune vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes nur wenn >1 Subcluster
    # -------------------------
    n_subclusters = sub.obs["leiden"].nunique()

    if n_subclusters >= 2:
        sc.tl.rank_genes_groups(
            sub,
            groupby="leiden",
            method="wilcoxon",
            n_genes=n_ranked
        )

        sc.pl.rank_genes_groups_dotplot(
            sub,
            n_genes=5,
            standard_scale="var",
            cmap="Reds",
            title=f"Cluster {cluster_id}: top genes",
            dendrogram=False
        )

        ranked = sc.get.rank_genes_groups_df(sub, group=None)

        print("\n=== Top ranked genes per subcluster ===")
        for scl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
            top_cl = ranked[ranked["group"] == scl].head(n_ranked)
            print(f"\nSubcluster {scl}")
            print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    else:
        ranked = None
        print("\nNur ein Subcluster gefunden -> keine ranked genes zwischen Subclustern möglich.")

    return sub, ranked

In [ ]:
# =========================================================
# Clusters selected for review within the immune compartment
# =========================================================
clusters_to_check = [
    "0", "2", "3", "4", "5", "7", "8"
]

cluster_resolutions = {
    "0": 0.6,
    "2": 0.6,
    "3": 0.6,
    "4": 0.5,
    "5": 0.6,
    "7": 0.6,
    "8": 0.6,
}

subclusters = {}
ranked_tables = {}

immune_sub.obs["leiden"] = immune_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(immune_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in immune_sub.obs.columns:
        immune_sub.obs[subcol] = pd.Series(index=immune_sub.obs_names, dtype="object")

    immune_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
# =========================================================
# Clusters selected for review within the immune compartment
# =========================================================
clusters_to_check = [
    "9", "12", "13", "14", "15", "16", "17"
]

cluster_resolutions = {
    "9": 0.5,
    "12": 0.5,
    "13": 0.5,
    "14": 0.5,
    "15": 0.5,
    "16": 0.5,
    "17": 0.5,
}

subclusters = {}
ranked_tables = {}

immune_sub.obs["leiden"] = immune_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(immune_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in immune_sub.obs.columns:
        immune_sub.obs[subcol] = pd.Series(index=immune_sub.obs_names, dtype="object")

    immune_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
#subclusters to small, rerun analysis

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
immune_sub = adata[adata.obs["celltype_corrected"] == "immune"].copy()
immune_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(immune_sub, target_sum=1e4)
sc.pp.log1p(immune_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(immune_sub, n_comps=30)
sc.pp.neighbors(immune_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(immune_sub, resolution=0.9, random_state=0, flavor="leidenalg")
sc.tl.umap(immune_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], immune_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], immune_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], immune_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], immune_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring ---
immune_sub.obs["CM_mean"] = np.array(immune_sub[:, cm_markers].X.mean(axis=1)).flatten()
immune_sub.obs["FB_mean"] = np.array(immune_sub[:, fb_markers].X.mean(axis=1)).flatten()
immune_sub.obs["Vasc_mean"] = np.array(immune_sub[:, vasc_markers].X.mean(axis=1)).flatten()
immune_sub.obs["Immune_mean"] = np.array(immune_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores ---
sc.tl.score_genes(immune_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(immune_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(immune_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(immune_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (immune-focused) ---
immune_sub.obs["Immune_minus_CM"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["CM_mean"]
immune_sub.obs["Immune_minus_FB"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["FB_mean"]
immune_sub.obs["Immune_minus_Vasc"] = immune_sub.obs["Immune_mean"] - immune_sub.obs["Vasc_mean"]

immune_sub.obs["CM_minus_Immune"] = immune_sub.obs["CM_mean"] - immune_sub.obs["Immune_mean"]
immune_sub.obs["FB_minus_Immune"] = immune_sub.obs["FB_mean"] - immune_sub.obs["Immune_mean"]
immune_sub.obs["Vasc_minus_Immune"] = immune_sub.obs["Vasc_mean"] - immune_sub.obs["Immune_mean"]

# --- cluster summary ---
immune_scores = (
    immune_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "Immune_minus_CM", "Immune_minus_FB", "Immune_minus_Vasc"
    ]]
    .mean()
    .sort_values("Immune_minus_CM", ascending=False)
)

print(immune_scores.round(3))

# --- visualization ---
sc.pl.umap(
    immune_sub,
    color=[
        "leiden",
        "Immune_mean",
        "CM_mean",
        "FB_mean",
        "Vasc_mean",
        "Immune_minus_CM",
        "Immune_minus_Vasc"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================
# Marker sets (immune context)
# =========================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "NKG7", "CD68", "FCGR3A", "MPEG1"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

marker_list = []
for genes in markers.values():
    marker_list += present(genes, immune_sub)

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    immune_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Immune subset: marker genes (immune / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    immune_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    immune_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="Immune subset: top ranked genes per cluster",
    dendrogram=False
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = immune_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    immune_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Immune subset: markers + ranked genes",
    dendrogram=False
)

In [ ]:
sc.tl.rank_genes_groups(
    immune_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    immune_sub,
    sharey=False,
    title="Ranked genes"
)

In [ ]:
import scanpy as sc
import pandas as pd

# =========================================================
# Marker sets for immune-cell subclustering
# =========================================================
markers = {
    "Immune_core": ["PTPRC", "CD3D", "CD3E", "NKG7", "CD68", "FCGR3A", "MPEG1"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB_contamination": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=15):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, random_state=0, flavor="leidenalg")
    sc.tl.umap(sub)

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    print("\nSubcluster sizes:")
    print(sub.obs["leiden"].value_counts())

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: immune vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes nur wenn >1 Subcluster
    # -------------------------
    n_subclusters = sub.obs["leiden"].nunique()

    if n_subclusters >= 2:
        sc.tl.rank_genes_groups(
            sub,
            groupby="leiden",
            method="wilcoxon",
            n_genes=n_ranked
        )

        sc.pl.rank_genes_groups_dotplot(
            sub,
            n_genes=5,
            standard_scale="var",
            cmap="Reds",
            title=f"Cluster {cluster_id}: top genes",
            dendrogram=False
        )

        ranked = sc.get.rank_genes_groups_df(sub, group=None)

        print("\n=== Top ranked genes per subcluster ===")
        for scl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
            top_cl = ranked[ranked["group"] == scl].head(n_ranked)
            print(f"\nSubcluster {scl}")
            print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    else:
        ranked = None
        print("\nNur ein Subcluster gefunden -> keine ranked genes zwischen Subclustern möglich.")

    return sub, ranked

In [ ]:
# =========================================================
# Clusters selected for review within the immune compartment
# =========================================================
clusters_to_check = [
    "3", "4", "5", "6", "7", "8", "9", "10", "11"
]

cluster_resolutions = {
    "3": 0.5,
    "4": 0.5,
    "5": 0.4,
    "6": 0.5,
    "7": 0.5,
    "8": 0.4,
    "9": 0.4,
    "10": 0.4,
    "11": 0.2,
}

subclusters = {}
ranked_tables = {}

immune_sub.obs["leiden"] = immune_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(immune_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in immune_sub.obs.columns:
        immune_sub.obs[subcol] = pd.Series(index=immune_sub.obs_names, dtype="object")

    immune_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
# =========================================================
# 0) Preparation
# =========================================================
immune_sub.obs["leiden"] = immune_sub.obs["leiden"].astype(str)

# Initialize all cells as immune before applying the manual mapping
immune_sub.obs["celltype_immune_refined"] = "immune"

# =========================================================
# 1) Subcluster-Mappings
# =========================================================
immune_maps = {
    "0": {
        "0": "immune",
        "1": "contaminated",
        "2": "contaminated",
        "3": "contaminated",
    },
    "2": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "contaminated",
    },
    "3": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "immune",
    },
    "4": {
        "0": "contaminated",
        "1": "CM",
        "2": "immune",
        "3": "contaminated",
    },
    "5": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "immune",
        "3": "contaminated",
    },
    "6": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "contaminated",
    },
    "7": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "contaminated",
    },
    "8": {
        "0": "contaminated",
        "1": "contaminated",
        "2": "contaminated",
    },
    "9": {
        "0": "immune",
        "1": "contaminated",
        "2": "contaminated",
    },
    "10": {
        "0": "immune",
        "1": "FB",
        "2": "contaminated",
    },
    "11": {
        "0": "FB",
        "1": "contaminated",
        "2": "contaminated",
    },
}

# =========================================================
# 2) Subcluster-Reannotation anwenden
# =========================================================
for main_clust, mapping in immune_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = immune_sub.obs["leiden"] == main_clust

    if subcol not in immune_sub.obs.columns:
        print(f"Warning: {subcol} not found. Skipping cluster {main_clust}.")
        continue

    immune_sub.obs.loc[mask, "celltype_immune_refined"] = (
        immune_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# =========================================================
# 3) Kontrolle in immune_sub
# =========================================================
print("=== Hauptcluster vs. immune refined annotation ===")
print(pd.crosstab(immune_sub.obs["leiden"], immune_sub.obs["celltype_immune_refined"]))

print("\n=== Finale Annotationen in immune_sub ===")
print(immune_sub.obs["celltype_immune_refined"].value_counts(dropna=False))

# =========================================================
# 4) Auf adata übertragen
# WICHTIG: bestehende celltype_corrected NICHT neu initialisieren
# =========================================================
common_cells = adata.obs_names.intersection(immune_sub.obs_names)

adata.obs.loc[common_cells, "celltype_corrected"] = (
    immune_sub.obs.loc[common_cells, "celltype_immune_refined"].astype(str)
)

# =========================================================
# 5) Kategorien setzen
# =========================================================
adata.obs["celltype_corrected"] = pd.Categorical(
    adata.obs["celltype_corrected"].astype(str),
    categories=[
        "CM",
        "FB",
        "vascular",
        "immune",
        "contaminated",
    ],
)

# =========================================================
# 6) Kontrolle in adata
# =========================================================
print("\n=== Verteilung in adata nach immune-Korrektur ===")
print(adata.obs["celltype_corrected"].value_counts(dropna=False))

print("\n=== Vergleich vorher vs. nach immune-Korrektur ===")
print(pd.crosstab(
    adata.obs.loc[common_cells, "celltype"],
    adata.obs.loc[common_cells, "celltype_corrected"]
))

In [ ]:
adata.write("DCM_second_round_v4.h5ad")

In [ ]:
adata = sc.read("DCM_second_round_v4.h5ad")

In [ ]:
print(adata)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# =========================================================
# 0) Marker sets
# =========================================================
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], adata)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], adata)
vasc_markers = present(["PECAM1", "VWF", "CD93"], adata)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], adata)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# =========================================================
# 1) Global expression-based scores auf adata
# =========================================================
adata.obs["CM_mean"] = np.array(adata[:, cm_markers].X.mean(axis=1)).flatten()
adata.obs["FB_mean"] = np.array(adata[:, fb_markers].X.mean(axis=1)).flatten()
adata.obs["Vasc_mean"] = np.array(adata[:, vasc_markers].X.mean(axis=1)).flatten()
adata.obs["Immune_mean"] = np.array(adata[:, imm_markers].X.mean(axis=1)).flatten()

# =========================================================
# 2) Dominanter Zelltyp + Margin berechnen
# =========================================================
score_cols = ["CM_mean", "FB_mean", "Vasc_mean", "Immune_mean"]

score_to_label = {
    "CM_mean": "CM",
    "FB_mean": "FB",
    "Vasc_mean": "vascular",
    "Immune_mean": "immune",
}

scores = adata.obs[score_cols]

adata.obs["dominant_score"] = scores.max(axis=1)
adata.obs["dominant_score_label"] = scores.idxmax(axis=1).map(score_to_label)

sorted_scores = np.sort(scores.values, axis=1)
adata.obs["score_margin"] = sorted_scores[:, -1] - sorted_scores[:, -2]
adata.obs["score_ratio"] = sorted_scores[:, -1] / (sorted_scores[:, -2] + 1e-6)

# =========================================================
# 3) QC: Wie sehen contaminated Zellen aus?
# =========================================================
print("=== celltype_corrected Verteilung ===")
print(adata.obs["celltype_corrected"].value_counts(dropna=False))

print("\n=== Dominanter Score innerhalb contaminated ===")
print(pd.crosstab(
    adata.obs["celltype_corrected"],
    adata.obs["dominant_score_label"]
))

print("\n=== Mean scores pro aktueller Annotation ===")
print(
    adata.obs
    .groupby("celltype_corrected", observed=False)[score_cols + ["score_margin", "score_ratio"]]
    .mean()
    .round(3)
)

# =========================================================
# 4) Kandidaten fürs Rescue anschauen
# =========================================================
contam = adata.obs["celltype_corrected"] == "contaminated"

rescue_summary = (
    adata.obs.loc[contam]
    .groupby("dominant_score_label")[["dominant_score", "score_margin", "score_ratio"]]
    .agg(["count", "mean", "median"])
    .round(3)
)

print("\n=== Rescue-Kandidaten innerhalb contaminated ===")
print(rescue_summary)

### Interpretation of the `contaminated` label

`contaminated` is an operational quality-control label for clusters with ambiguous or mixed lineage-marker patterns, rather than a biological cell type. A single spatial cluster can contain cells expressing markers associated with more than one lineage, or a subset of cells can show discordant marker profiles. These cells are set aside for a separate reclustering and marker review so that cells with coherent cardiomyocyte, fibroblast or immune signatures can be reassigned where supported by the marker evidence. Cells that remain ambiguous should retain the `contaminated` label; this label alone does not establish the cause of the mixed signal (for example, segmentation error, doublets or biological co-expression).


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# =========================================================
# --- Subset cells labelled as contaminated ---
# =========================================================
contam_sub = adata[adata.obs["celltype_corrected"] == "contaminated"].copy()
contam_sub

# =========================================================
# --- preprocessing (Xenium-friendly) ---
# =========================================================
sc.pp.normalize_total(contam_sub, target_sum=1e4)
sc.pp.log1p(contam_sub)

# =========================================================
# --- Clustering (no highly variable gene selection or scaling) ---
# =========================================================
sc.tl.pca(contam_sub, n_comps=30)
sc.pp.neighbors(contam_sub, n_neighbors=15, n_pcs=30)

# 🔥 wichtig: höher auflösen als vorher
sc.tl.leiden(contam_sub, resolution=1.8, random_state=0, flavor="leidenalg")

sc.tl.umap(contam_sub)

# =========================================================
# --- marker sets ---
# =========================================================
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], contam_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], contam_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], contam_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], contam_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# =========================================================
# --- expression-based scoring ---
# =========================================================
contam_sub.obs["CM_mean"] = np.array(contam_sub[:, cm_markers].X.mean(axis=1)).flatten()
contam_sub.obs["FB_mean"] = np.array(contam_sub[:, fb_markers].X.mean(axis=1)).flatten()
contam_sub.obs["Vasc_mean"] = np.array(contam_sub[:, vasc_markers].X.mean(axis=1)).flatten()
contam_sub.obs["Immune_mean"] = np.array(contam_sub[:, imm_markers].X.mean(axis=1)).flatten()

# optional scanpy scores
sc.tl.score_genes(contam_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(contam_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(contam_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(contam_sub, imm_markers, score_name="Immune_score")

# =========================================================
# --- derived metrics ---
# =========================================================
contam_sub.obs["Immune_minus_CM"] = contam_sub.obs["Immune_mean"] - contam_sub.obs["CM_mean"]
contam_sub.obs["Immune_minus_FB"] = contam_sub.obs["Immune_mean"] - contam_sub.obs["FB_mean"]
contam_sub.obs["Immune_minus_Vasc"] = contam_sub.obs["Immune_mean"] - contam_sub.obs["Vasc_mean"]

contam_sub.obs["CM_minus_Immune"] = contam_sub.obs["CM_mean"] - contam_sub.obs["Immune_mean"]
contam_sub.obs["FB_minus_Immune"] = contam_sub.obs["FB_mean"] - contam_sub.obs["Immune_mean"]
contam_sub.obs["Vasc_minus_Immune"] = contam_sub.obs["Vasc_mean"] - contam_sub.obs["Immune_mean"]

# =========================================================
# 🔥 WICHTIG: Cluster-Level Summary (entscheidend!)
# =========================================================
cluster_summary = (
    contam_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "Immune_minus_CM", "Immune_minus_FB", "Immune_minus_Vasc"
    ]]
    .mean()
    .sort_values("CM_mean", ascending=False)
)

print("\n=== Cluster summary (contaminated reclustering) ===")
print(cluster_summary.round(3))

# =========================================================
# --- optional: dominant score je cluster ---
# =========================================================
cluster_summary["dominant"] = cluster_summary[
    ["CM_mean", "FB_mean", "Vasc_mean", "Immune_mean"]
].idxmax(axis=1)

print("\n=== Dominanter Zelltyp pro Cluster ===")
print(cluster_summary["dominant"])

# =========================================================
# --- visualization ---
# =========================================================
sc.pl.umap(
    contam_sub,
    color=[
        "leiden",
        "CM_mean",
        "FB_mean",
        "Vasc_mean",
        "Immune_mean",
        "Immune_minus_CM",
        "Immune_minus_Vasc"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================================================
# Marker sets (contaminated context)
# =========================================================
markers = {
    "CM_signal": ["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"],
    "FB_signal": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular_signal": ["PECAM1", "VWF", "CD93"],
    "Immune_signal": ["PTPRC", "CD3D", "CD3E", "NKG7", "CD68", "FCGR3A", "MPEG1"]
}

# Keep only genes present in the dataset
marker_list = []
for genes in markers.values():
    marker_list += present(genes, contam_sub)

# Duplikate entfernen (wichtig!)
marker_list = list(dict.fromkeys(marker_list))

# =========================================================
# 1. Marker Dotplot
# =========================================================
sc.pl.dotplot(
    contam_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Contaminated recluster: marker genes"
)

# =========================================================
# 2. Rank genes
# =========================================================
sc.tl.rank_genes_groups(
    contam_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================================================
# 3. Ranked genes dotplot
# =========================================================
sc.pl.rank_genes_groups_dotplot(
    contam_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="Contaminated recluster: top ranked genes",
    dendrogram=False
)

# =========================================================
# 4. Combined view (Marker + ranked)
# =========================================================
ranked = contam_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    contam_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="Contaminated recluster: markers + ranked genes",
    dendrogram=False
)

In [ ]:
sc.tl.rank_genes_groups(
    contam_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    contam_sub,
    sharey=False,
    title="Ranked genes"
)

In [ ]:
# =========================================================
# 0) Preparation
# =========================================================
contam_sub.obs["leiden"] = contam_sub.obs["leiden"].astype(str)

# Initialize all cells as contaminated before applying the manual mapping
contam_sub.obs["celltype_contam_refined"] = "contaminated"

# =========================================================
# 1) Direkte Reannotation nach contaminated reclustering
# =========================================================

# IMMUNE
contam_sub.obs.loc[
    contam_sub.obs["leiden"] == "22",
    "celltype_contam_refined"
] = "immune"

# FB
contam_sub.obs.loc[
    contam_sub.obs["leiden"].isin(["2", "9", "10", "12"]),
    "celltype_contam_refined"
] = "FB"

# CM
contam_sub.obs.loc[
    contam_sub.obs["leiden"].isin(["6", "17", "23"]),
    "celltype_contam_refined"
] = "CM"

# =========================================================
# 2) Kontrolle in contam_sub
# =========================================================
print("=== Hauptcluster vs. contaminated refined annotation ===")
print(pd.crosstab(contam_sub.obs["leiden"], contam_sub.obs["celltype_contam_refined"]))

print("\n=== Finale Annotationen in contam_sub ===")
print(contam_sub.obs["celltype_contam_refined"].value_counts(dropna=False))

# =========================================================
# 3) Auf adata übertragen
# WICHTIG: bestehende celltype_corrected NICHT neu initialisieren
# =========================================================
common_cells = adata.obs_names.intersection(contam_sub.obs_names)

adata.obs.loc[common_cells, "celltype_corrected"] = (
    contam_sub.obs.loc[common_cells, "celltype_contam_refined"].astype(str)
)

# =========================================================
# 4) Kategorien setzen
# =========================================================
adata.obs["celltype_corrected"] = pd.Categorical(
    adata.obs["celltype_corrected"].astype(str),
    categories=[
        "CM",
        "FB",
        "vascular",
        "immune",
        "contaminated",
    ],
)

# =========================================================
# 5) Kontrolle in adata
# =========================================================
print("\n=== Verteilung in adata nach contaminated-Rescue ===")
print(adata.obs["celltype_corrected"].value_counts(dropna=False))

print("\n=== Vergleich vorher vs. nach contaminated-Rescue ===")
print(pd.crosstab(
    adata.obs.loc[common_cells, "celltype"],
    adata.obs.loc[common_cells, "celltype_corrected"]
))

In [ ]:
adata.write("DCM_final.h5ad")

In [ ]:
# Save the final DCM annotation after compartment-specific refinement.
DCM_FINAL = Path("DCM_final.h5ad")
adata.write_h5ad(DCM_FINAL, compression="gzip")
print(f"Saved final DCM annotation: {DCM_FINAL.resolve()}")
print(adata.obs["celltype_corrected"].value_counts(dropna=False))

### DCM final-object check

In [ ]:
from pathlib import Path
if Path("DCM_final.h5ad").exists():
    dcm_final_check = sc.read_h5ad("DCM_final.h5ad")
    print(dcm_final_check)
    if "celltype_corrected" in dcm_final_check.obs:
        print(dcm_final_check.obs["celltype_corrected"].value_counts(dropna=False))
    else:
        print("Warning: celltype_corrected is absent from DCM_final.h5ad")
else:
    print("DCM_final.h5ad has not yet been written. Run the DCM refinement cells above in order.")

## Part C — IHD: reconstruct counts, QC, BANKSY clustering and broad annotation

## 1. Imports and parameters

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData

from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy.cluster_methods import run_Leiden_partition
from banksy_utils.umap_pca import pca_umap

RANDOM_SEED = 1234
CELL_FILE = Path("Clean_cells_IHD.csv")
UMI_FILE = Path("Clean_UMIs_IHD.csv")
OUTPUT_FILE = Path("adata_IHD_annotated.h5ad")

for required in [CELL_FILE, UMI_FILE]:
    if not required.exists():
        raise FileNotFoundError(f"Required input file not found: {required.resolve()}")
sc.settings.verbosity = 2

## 2. Reconstruct AnnData and attach spatial coordinates

In [ ]:
df_cells = pd.read_csv(CELL_FILE)
df_umi = pd.read_csv(UMI_FILE)

required_umi_cols = {"Cell_code", "Gene", "UMI"}
missing = required_umi_cols - set(df_umi.columns)
if missing:
    raise KeyError(f"Missing columns in {UMI_FILE.name}: {sorted(missing)}")
if not {"x_centroid", "y_centroid"}.issubset(df_cells.columns):
    raise KeyError("Clean_cells_IHD.csv must contain x_centroid and y_centroid.")

expr = df_umi.pivot_table(
    index="Cell_code", columns="Gene", values="UMI", aggfunc="sum", fill_value=0
)
if "cell_id" in df_cells.columns:
    df_cells["cell_id"] = df_cells["cell_id"].astype(str)
    df_cells = df_cells.set_index("cell_id", drop=True)
else:
    df_cells.index = df_cells.index.astype(str)
    df_cells.index.name = "cell_id"
expr.index = expr.index.astype(str)

common = expr.index.intersection(df_cells.index)
if len(common) == 0:
    raise ValueError("No shared cell IDs between expression and cell-coordinate tables.")
expr = expr.loc[common].copy()
df_cells = df_cells.loc[common].copy()
coords = df_cells[["x_centroid", "y_centroid"]].to_numpy(dtype=float)

adata = AnnData(
    X=expr.to_numpy(dtype=np.float32),
    obs=pd.DataFrame(index=expr.index.astype(str)),
    var=pd.DataFrame(index=expr.columns.astype(str)),
)
adata.obsm["coord_xy"] = coords
adata.obsm["spatial"] = coords
adata.obs["x_centroid"] = coords[:, 0]
adata.obs["y_centroid"] = coords[:, 1]
print("AnnData:", adata)

## 3. Quality control

The IHD notebook used the 5th percentile as the low cutoff and the 99.5th percentile as the high cutoff for both counts and detected genes.

In [ ]:
adata.obs["n_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs["n_genes"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()

count_low = float(np.percentile(adata.obs["n_counts"], 5))
gene_low = float(np.percentile(adata.obs["n_genes"], 5))
count_high = float(np.percentile(adata.obs["n_counts"], 99.5))
gene_high = float(np.percentile(adata.obs["n_genes"], 99.5))

adata.obs["keep_qc"] = (
    (adata.obs["n_counts"] >= count_low)
    & (adata.obs["n_counts"] <= count_high)
    & (adata.obs["n_genes"] >= gene_low)
    & (adata.obs["n_genes"] <= gene_high)
)
print({
    "n_before": int(adata.n_obs),
    "n_after": int(adata.obs["keep_qc"].sum()),
    "n_removed": int((~adata.obs["keep_qc"]).sum()),
    "count_low_5pct": count_low,
    "gene_low_5pct": gene_low,
    "count_high_99_5pct": count_high,
    "gene_high_99_5pct": gene_high,
})
adata = adata[adata.obs["keep_qc"].to_numpy()].copy()
sc.pl.violin(adata, ["n_counts", "n_genes"], jitter=0.4, multi_panel=True)

## 4. BANKSY and Leiden clustering

Parameters used for the initial IHD clustering: 15 neighbours, `lambda=0.2`, 30 PCA dimensions, and resolution 1.0 for the initial cell-type assignment.

In [ ]:
coord_keys = ("x_centroid", "y_centroid", "coord_xy")
banksy_dict = initialize_banksy(
    adata=adata,
    coord_keys=coord_keys,
    num_neighbours=15,
    nbr_weight_decay="scaled_gaussian",
    max_m=1,
    plt_edge_hist=False,
    plt_nbr_weights=False,
    plt_agf_angles=False,
    plt_theta=False,
)
banksy_dict, banksy_matrix = generate_banksy_matrix(
    adata, banksy_dict, lambda_list=[0.2], max_m=1
)
pca_umap(banksy_dict, pca_dims=[30], add_umap=True)

banksy_df_10, _ = run_Leiden_partition(
    banksy_dict, resolutions=[1.0], num_nn=50, num_iterations=-1,
    partition_seed=RANDOM_SEED, match_labels=True, max_labels=50,
)

adata_banksy = banksy_dict["scaled_gaussian"][0.2]["adata"]
adata.obsm["X_umap"] = adata_banksy.obsm["reduced_pc_30_umap"]
adata.obsm["spatial"] = np.asarray(adata.obsm["coord_xy"])
key_10 = banksy_df_10.index[0]
labels = np.asarray(banksy_df_10.labels[key_10].dense).astype(int)
if len(labels) != adata.n_obs:
    raise ValueError("BANKSY labels do not match AnnData observations.")
adata.obs["banksy_cluster_1.0"] = pd.Categorical(labels.astype(str))
print(adata.obs["banksy_cluster_1.0"].value_counts().sort_index())
sc.pl.embedding(adata, basis="X_umap", color="banksy_cluster_1.0", size=6,
                legend_loc="on data", title="BANKSY clusters (resolution 1.0, UMAP)")
sc.pl.embedding(adata, basis="spatial", color="banksy_cluster_1.0", size=6,
                title="BANKSY clusters (resolution 1.0, spatial)")

## 5. Marker-based annotation review

In [ ]:
marker_genes = {
    "CM": ["TNNT2", "MYH6", "MYH7"],
    "FB": ["COL3A1", "LUM", "PDGFRA", "DDR2"],
    "Endo": ["PECAM1", "VWF", "ACKR1"],
    "Myeloid": ["CD68", "ADGRE1"],
    "Tcell": ["CD3D", "CD3E", "TRAC"],
    "Bcell": ["MS4A1", "CD79A"],
    "Adipocyte": ["ADIPOQ"],
    "Vascular_SMC": ["ACTA2", "TAGLN", "MYH11"],
}
marker_list = sorted({g for genes in marker_genes.values() for g in genes if g in adata.var_names})
if marker_list:
    sc.pl.dotplot(adata, marker_list, groupby="banksy_cluster_1.0", standard_scale="var", cmap="Reds")

adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)
sc.tl.rank_genes_groups(adata_norm, groupby="banksy_cluster_1.0", method="wilcoxon")
sc.pl.rank_genes_groups(adata_norm, n_genes=30, sharey=False, title="Ranked genes (resolution 1.0)")
sc.pl.rank_genes_groups_dotplot(adata_norm, groupby="banksy_cluster_1.0", n_genes=5,
                               standard_scale="var", cmap="Reds", dendrogram=False)

## 6. Initial manual cluster-to-cell-type mapping

The following manual mapping assigns the BANKSY clusters to broad cell types. Cluster 7 was labelled `low_quality` and removed before the spatial-proximity analyses.

In [ ]:
cluster_to_celltype = {
    0: "FB",
    1: "vascular",
    2: "CM",
    3: "CM",
    4: "CM",
    5: "vascular",
    6: "FB",
    7: "low_quality",
    8: "immune",
    9: "vascular",
    10: "vascular",
    11: "CM",
    12: "FB",
    13: "immune",
    14: "FB",
    15: "CM",
    16: "vascular",
    17: "immune",
    18: "vascular",
    19: "immune",
    20: "vascular",
}
clusters = adata.obs["banksy_cluster_1.0"].astype(int)
adata.obs["celltype"] = clusters.map(cluster_to_celltype)
if adata.obs["celltype"].isna().any():
    unmapped = sorted(clusters[adata.obs["celltype"].isna()].unique().tolist())
    raise ValueError(f"BANKSY clusters lack a cell-type mapping: {unmapped}")
adata = adata[adata.obs["celltype"] != "low_quality"].copy()
adata.obs["celltype"] = pd.Categorical(adata.obs["celltype"], categories=["CM", "FB", "vascular", "immune"])
print(adata.obs["celltype"].value_counts(dropna=False))
sc.pl.umap(adata, color="celltype")
sc.pl.embedding(adata, basis="spatial", color="celltype", size=6)

## 7. Save first-pass IHD annotations

In [ ]:
adata.write_h5ad(OUTPUT_FILE, compression="gzip")
print(f"Saved: {OUTPUT_FILE.resolve()}")

## Outputs and next step

- DCM final annotated object: `DCM_final.h5ad`, with `celltype_corrected`.
- IHD first-pass object: `adata_IHD_annotated.h5ad`.

Continue with `02_IHD_compartment_refinement_to_v3.ipynb` to reproduce the manual fibroblast, cardiomyocyte, vascular and immune refinement that creates `adata_celltype_corrected_v3.h5ad`. Then run `03_IHD_final_fibroblast_refinement.ipynb`.